# PQCBM: learning a distribution of correlated bitstrings

This small, reproducible experiment introduces `create_qcbm`, expectation-based Gaussian MMD, differentiable Pauli propagation, direct device sampling, and Quantikz export. We learn the equal mixture of `000` and `111` using three qubits and two YZ/CNOT-chain layers.

The objective follows the Pauli-Z expectation-mixture identity of [Recio-Armengol, Ahmed and Bowles (2025), §4.2](https://arxiv.org/abs/2503.02934). Here we enumerate all eight masks to make the objective exact. The scalable interface samples masks; neither mode needs a full probability vector for training. Unpruned PP is used for comparison, not as a claim of efficient simulation for arbitrary circuits.

Run the cells in order from the repository or its `examples` directory, with QASP, Torch, NumPy, Matplotlib and PennyLane installed. See [the module guide](../docs/genlearn.md) for all options and limitations.

In [ ]:
from pathlib import Path
import sys
import json
import numpy as np
import torch
from IPython.display import display, Image, FileLink

ROOT = Path.cwd() if (Path.cwd() / "qasp").exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT))
from qasp import create_qcbm, draw_qcbm
from examples.pqcbm_generative_learning import CONFIG, OUTPUT, run_experiment

torch.set_num_threads(1)
print(json.dumps(CONFIG, indent=2))

## Construct and inspect a model

The dictionary uses the same component names and `ansatz_config` convention as other QASP modules. `include_imprimitive=True` enables the entangling CNOT gates. A generative model needs no feature map unless we want fixed conditioning inputs. Data are target output bitstrings, with the leftmost column corresponding to wire zero.

In [ ]:
data = np.array([[0, 0, 0], [1, 1, 1]] * 128)
model = create_qcbm(CONFIG, data=data)
print("Parameter shape:", tuple(model.params.shape))
print("Initial MMD squared:", float(model.loss().detach()))
print(model.draw(output="text"))

## What the MMD objective measures

For a bit mask $a$, the target moment is the empirical parity $\mu_p(a)=N^{-1}\sum_x(-1)^{x\cdot a}$, while the model moment is $\langle Z_a\rangle_\theta$. With Gaussian standard deviation $\sigma$, bits of a sampled mask are independently one with probability $(1-e^{-1/(2\sigma^2)})/2$.

The loss is a weighted mean of squared moment differences. Multiple bandwidths define a uniform mixture of kernels. The identity mask has zero discrepancy but retains its original weight. We use the empirical target convention; `estimator="unbiased"` removes the data diagonal term. The reference PDF v1 prints inconsistent bandwidth notation between Eqs. (3) and (17); QASP consistently uses the standard-deviation convention from Eq. (3).

In [ ]:
target_moments, _ = model.mmd.target_statistics(data)
quantum_moments = model.moments().detach().numpy()
print("mask   target    model   kernel weight")
for mask, target, predicted, weight in zip(model.mmd.masks, target_moments,
                                         quantum_moments, model.mmd.weights):
    print("".join(map(str, mask)), f" {target: .5f}  {predicted: .5f}  {weight:.5f}")
assert np.isclose(model.mmd.weights.sum(), 1)

## Train and compare two backends

The companion script runs 120 Adam updates from identical initial parameters on `default.qubit` and `lightning.pp` with no pruning or weight truncation. It checks initial gradients and final probability diagnostics, saves the learned parameters, generates a Quantikz source file, and draws 2,000 samples using a sampling device.

The code uses `model.train(steps)`, returning parameters and pre-update losses. Runtimes below are for this tiny experiment and are not a performance benchmark.

In [ ]:
summary = run_experiment(OUTPUT, steps=120)
print(json.dumps(summary, indent=2))
display(Image(filename=str(OUTPUT / "learning.png")))

## Restore parameters and sample the learned circuit

Pauli propagation only provides expectations. We explicitly choose a sampling backend for deployment, using the same trained circuit and parameters. `sample()` executes a sampling measurement; it does not first enumerate probabilities. The returned array has one bitstring per row.

In [ ]:
saved = json.loads((OUTPUT / "trained_parameters.json").read_text())
restored = create_qcbm({**saved["config"], "device": "lightning.pp",
                        "init_params": saved["params"], "pp_prune": False})
bits = restored.sample(12, device="default.qubit", device_kwargs={"seed": 19})
print(bits)
assert bits.shape == (12, 3)
print("PP diagnostic MMD:", float(restored.loss(data).detach()))

## Sampled observables and finite shots

Change `mode` to `sampled` for an observable budget independent of $2^n$. The trainer resamples masks once per step, keeping them fixed during all evaluations for the gradient. For a reproducible evaluation objective, keep its masks fixed or use exact MMD at small width.

Finite-shot devices can use `shots=...` with `diff_method="parameter-shift"`; the model-square contribution uses two independent expectation evaluations. PP instead requires analytic execution and uses Torch backpropagation.

In [ ]:
sampled = create_qcbm({**CONFIG, "device": "lightning.pp", "pp_prune": False,
                       "mmd_config": {"mode": "sampled", "n_observables": 32,
                                      "bandwidths": [0.5, 1.0, 2.0]}}, data=data)
_, short_history = sampled.train(5)
print("Sampled-mask MMD losses:", short_history)
assert np.isfinite(short_history).all()

## Paper-ready drawing

Quantikz source is editable and compiles to a vector PDF with a local LaTeX installation containing Quantikz. `standalone=False` produces a snippet for a manuscript; `compress=True` expands the first layer and abbreviates repeated layers. This is the quantum sampling model; MMD aggregation happens classically.

In [ ]:
tex = draw_qcbm(restored, output="tikz", tikz_options={
    "standalone": True, "symbolic_parameters": True,
    "compress": True, "gate_colors": False,
})
assert "quantikz" in tex
print(tex)
display(FileLink(str(OUTPUT / "pqcbm.tex")))
if (OUTPUT / "pqcbm.png").exists():
    display(Image(filename=str(OUTPUT / "pqcbm.png")))

## Configure MPI execution

`mpi=True` requests one statevector distributed across ranks on an MPI-enabled
Lightning installation. Every rank must execute the same model calls with
identical data, seeds, batches and masks; only file writes should be restricted
to rank zero. The statevector still grows exponentially.

This cell only prepares configuration; it does not start MPI. Use the
[`pqcbm_mpi_smoke.py`](../tests/tests_deucalion/pqcbm_mpi_smoke.py) script for
collective training and reference checks on a cluster. Launch it with your
cluster's MPI launcher and allocated resources.


In [ ]:
cluster_config = {
    **CONFIG,
    "device": "lightning.kokkos",
    "mpi": True,
    "diff_method": "adjoint",
    "seed": 7,
    "mmd_config": {"mode": "sampled", "n_observables": 128,
                   "bandwidths": [0.5, 1.0, 2.0]},
}
print(json.dumps(cluster_config, indent=2))
# On every MPI rank: create_qcbm(cluster_config, data=data).train(20)


## Interpretation

The saved experiment reports its actual error, target-support mass, package versions and backend agreement. A successful result demonstrates the new API and numerically consistent training on a tiny circuit. It does not establish quantum advantage, efficient large-circuit PP, or a multi-node MPI run. QASP implements the paper's expectation-mixture MMD identity, not its specialized IQP Monte Carlo algorithm or thousand-qubit benchmark.

For NLL, fixed feature maps, natural gradients, device compatibility and migration from `BornMachine`, see [the complete module guide](../docs/genlearn.md).